In [ ]:
# ==============================================================================
# 🛠️ Étape 1 : Installation des dépendances et configuration de l'environnement
# ==============================================================================
%pip install peft==0.4.0
%pip install datasets transformers accelerate

import os
import time
import torch
import transformers
import peft
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, PeftModel

# Création du dossier cache si nécessaire
os.makedirs("cache", exist_ok=True)

# ==============================================================================
# 📦 Étape 2 : Chargement du Modèle de Fondation et du Tokenizer
# ==============================================================================
model_name = "bigscience/bloomz-560m"

# Chargement du tokenizer et du modèle causal de base
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

# ==============================================================================
# 📊 Étape 3 : Chargement et Prétraitement du Dataset (Échantillon de 10%)
# ==============================================================================
# Chargement du dataset "Abirate/english_quotes"
dataset = load_dataset("Abirate/english_quotes", split="train")

# Sélection aléatoire ou séquentielle d'un échantillon de 10%
# Le jeu de données complet contient environ 2508 lignes, 10% vaut ~250 lignes.
sample_size = int(0.10 * len(dataset))
shuffled_dataset = dataset.shuffle(seed=123)
data = shuffled_dataset.select(range(sample_size))

# Tokenization du texte des citations
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)

# Extraction d'un petit échantillon de validation/visualisation (5 exemples)
train_sample = data.select(range(5))
print("\n--- Aperçu de l'échantillon d'entraînement ---")
print(train_sample)

# ==============================================================================
# ⚙️ Étape 4 : Configuration et Application de LoRA (PEFT)
# ==============================================================================
# Remplacement et injection des matrices de bas rang dans les modules cibles de BLOOM
lora_config = LoraConfig(
    r=1,                          # Rang d'adaptation très faible pour l'exercice
    lora_alpha=1,                 # Facteur d'échelle ajustant la magnitude de la matrice
    target_modules=["query_key_value"], # Couches d'attention cibles pour les modèles BLOOM
    lora_dropout=0.05,
    bias="none",                  # Spécifie qu'aucun paramètre de biais ne sera entraîné
    task_type="CAUSAL_LM"
)

# Application de la configuration de l'adaptateur LoRA au modèle de fondation
peft_model = get_peft_model(foundation_model, lora_config)

print("\n--- Paramètres entraînables après injection LoRA ---")
peft_model.print_trainable_parameters()

# ==============================================================================
# 🏋️ Étape 5 : Paramètres d'Entraînement et Lancement du Trainer
# ==============================================================================
output_directory = os.path.join("cache/working", "peft_lab_outputs")

training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,    # Ajuste automatiquement la taille des lots selon la mémoire
    learning_rate=3e-2,           # Taux d'apprentissage plus élevé typique pour PEFT
    num_train_epochs=1,           # 1 époque pour des raisons de rapidité d'exécution
    use_cpu=True                  # Forcé sur CPU selon le template
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=data,           # Utilisation complète de l'échantillon des 10% tokenisé
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

print("\n--- Début du Fine-Tuning LoRA ---")
trainer.train()

# ==============================================================================
# 💾 Étape 6 : Sauvegarde de l'Adaptateur LoRA
# ==============================================================================
time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")

# Sauvegarde uniquement des poids légers de l'adaptateur (matrices A et B)
trainer.model.save_pretrained(peft_model_path)
print(f"\n✅ Modèle LoRA sauvegardé avec succès à l'emplacement : {peft_model_path}")

# ==============================================================================
# 🔮 Étape 7 : Chargement pour Inférence et Génération de Texte
# ==============================================================================
# Rechargement du modèle de base vierge pour prouver l'association
base_model = AutoModelForCausalLM.from_pretrained(model_name)

# Chargement combiné du modèle de base et de l'adaptateur fine-tuné
inference_model = PeftModel.from_pretrained(base_model, peft_model_path, is_trainable=False)

# Préparation d'une invite de texte
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")

# Génération des tokens de sortie
outputs = inference_model.generate(
    input_ids=inputs["input_ids"], 
    max_new_tokens=25,
    do_sample=True,
    top_k=50,
    top_p=0.95
)

print("\n--- Résultat de la génération de texte ---")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])